# 04-2. 연속 반복 정제

원본 summary는 수정하지 않는다. summary에서 **인접하여 반복되는 2어절 이상의 구간**만 첫 번째 구간을 남기고 제거한 별도 clean 데이터셋을 만든다. 멀리 떨어져 우연히 같은 4어절이 나온 경우는 보존한다.

In [1]:
import os
import sys
from pathlib import Path

PROJECT = Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert PROJECT.exists(), "프로젝트 경로를 확인하세요."
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), (
    "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
)

os.environ["HF_HOME"] = str(PROJECT / ".cache" / "huggingface")
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

DATASET_ID = "daekeun-ml/naver-news-summarization-ko"
CLEAN_DIR = PROJECT / "processed_data" / "repeat_4gram_cleaned"
COMPARE_DIR = PROJECT / "review_data" / "04-2_repeat_4gram_comparison"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
COMPARE_DIR.mkdir(parents=True, exist_ok=True)
print("정제 데이터:", CLEAN_DIR)
print("비교 데이터:", COMPARE_DIR)

정제 데이터: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\processed_data\repeat_4gram_cleaned
비교 데이터: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\review_data\04-2_repeat_4gram_comparison


In [2]:
import re

import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi

pd.set_option("display.max_colwidth", 100)

api = HfApi()
dataset_revision = api.dataset_info(DATASET_ID).sha
raw_dataset = load_dataset(DATASET_ID, revision=dataset_revision)

frames = []
for split in ["train", "validation", "test"]:
    frame = raw_dataset[split].to_pandas().copy()
    frame.insert(0, "source_index", np.arange(len(frame), dtype=np.int64))
    frame.insert(0, "split", split)
    frame.insert(0, "row_id", split + ":" + frame["source_index"].astype(str))
    frames.append(frame)

all_df = pd.concat(frames, ignore_index=True)
print("dataset revision:", dataset_revision)
display(all_df.groupby("split", sort=False).size().rename("rows").to_frame())

dataset revision: 916dfe6aed651b2ecd9d04fa0a704f8953e90762


,rows
split,
train,22194
validation,2466
test,2740


In [3]:
MIN_REPEAT_WORDS = 2
MAX_REPEAT_WORDS = 80

def remove_adjacent_repeats(text, min_words=MIN_REPEAT_WORDS, max_words=MAX_REPEAT_WORDS):
    """인접한 동일 어절 구간을 하나만 남긴다. 반환값은 (정제문, 제거 횟수)다."""
    tokens = str(text).split()
    removed_blocks = 0

    while True:
        found = False
        for start in range(len(tokens)):
            longest = min(max_words, (len(tokens) - start) // 2)
            for width in range(longest, min_words - 1, -1):
                if tokens[start:start + width] == tokens[start + width:start + 2 * width]:
                    # X X 또는 X X 일부(X)처럼 반복 끝이 잘린 경우도 두 번째 구간 전체를 제거합니다.
                    tail = 0
                    while (
                        tail < width
                        and start + 2 * width + tail < len(tokens)
                        and tokens[start + tail] == tokens[start + 2 * width + tail]
                    ):
                        tail += 1
                    del tokens[start + width:start + 2 * width + tail]
                    removed_blocks += 1
                    found = True
                    break
            if found:
                break
        if not found:
            return " ".join(tokens), removed_blocks

def has_any_repeat_4gram(text):
    tokens = str(text).split()
    grams = [tuple(tokens[i:i + 4]) for i in range(len(tokens) - 3)]
    return len(grams) != len(set(grams))

cleaned = all_df["summary"].map(remove_adjacent_repeats)
all_df["summary_clean"] = cleaned.map(lambda x: x[0])
all_df["removed_repeat_blocks"] = cleaned.map(lambda x: x[1]).astype(int)
all_df["summary_changed"] = all_df["summary"] != all_df["summary_clean"]
all_df["summary_words_before"] = all_df["summary"].str.split().str.len()
all_df["summary_words_after"] = all_df["summary_clean"].str.split().str.len()
all_df["removed_words"] = all_df["summary_words_before"] - all_df["summary_words_after"]
all_df["repeat_4gram_before"] = all_df["summary"].map(has_any_repeat_4gram)
all_df["repeat_4gram_after"] = all_df["summary_clean"].map(has_any_repeat_4gram)

summary = all_df.groupby("split", sort=False).agg(
    rows=("row_id", "size"),
    changed_rows=("summary_changed", "sum"),
    removed_blocks=("removed_repeat_blocks", "sum"),
    removed_words=("removed_words", "sum"),
    repeat_4gram_before=("repeat_4gram_before", "sum"),
    repeat_4gram_after=("repeat_4gram_after", "sum"),
).reset_index()
display(summary)

,split,rows,changed_rows,removed_blocks,removed_words,repeat_4gram_before,repeat_4gram_after
0,train,22194,2702,3021,28956,4325,3944
1,validation,2466,317,282,2572,454,410
2,test,2740,359,437,3909,517,467


In [4]:
# 정제된 split 전체와, 실제로 바뀐 요약의 전후 비교표를 분리해 저장합니다.
for split, frame in all_df.groupby("split", sort=False):
    output = frame.copy()
    output["summary"] = output["summary_clean"]
    output = output.drop(columns=["summary_clean"])
    output.to_csv(CLEAN_DIR / f"{split}.csv", index=False, encoding="utf-8-sig")

compare_columns = [
    "row_id", "split", "date", "category", "press", "title", "link",
    "document", "summary", "summary_clean", "summary_changed",
    "removed_repeat_blocks", "summary_words_before", "summary_words_after",
    "removed_words", "repeat_4gram_before", "repeat_4gram_after",
]
changed = all_df.loc[all_df["summary_changed"], compare_columns].copy()
changed["review_decision"] = ""
changed["memo"] = ""
changed.to_csv(COMPARE_DIR / "changed_summary_comparison.csv", index=False, encoding="utf-8-sig")
summary.to_csv(COMPARE_DIR / "cleaning_summary.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([{
    "dataset_id": DATASET_ID,
    "dataset_revision": dataset_revision,
    "rule": "remove only adjacent repeated spans of 2+ whitespace tokens; keep first span",
    "min_repeat_words": MIN_REPEAT_WORDS,
    "max_repeat_words": MAX_REPEAT_WORDS,
}]).to_csv(COMPARE_DIR / "run_info.csv", index=False, encoding="utf-8-sig")

print("저장 완료")
print("-", COMPARE_DIR / "changed_summary_comparison.csv")
print("-", CLEAN_DIR / "train.csv")
display(changed.head(10))

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\KDS-30\\Desktop\\Projects\\06_Team_Project4\\review_data\\04-2_repeat_4gram_comparison\\changed_summary_comparison.csv'

`changed_summary_comparison.csv`에서 원본 `summary`와 `summary_clean`을 직접 비교한다. 의미가 훼손된 사례가 있으면 다음 단계에서 해당 `row_id`를 원복 목록으로 관리한다. 정제 결과를 확인한 뒤에만 04-3을 실행한다.